In [1]:
import sys, os, re, json, time, glob
import pandas as pd
import numpy as np
import cv2
import pytesseract
import Levenshtein
from jiwer import wer

# reuse OCR-system code from week3 Lab3 (same engine, no re-implementation)
OCR_SYS_SRC = "../../week3/Lab3_ocr_system/Lab3_ocr_system/ocr_system/src"
sys.path.insert(0, OCR_SYS_SRC)
from ocr_system.evaluation import normalize_text, char_error_rate

DATASET = "../../week5/Lab5_transcript_dataset"
MANIFEST = f"{DATASET}/labels/manifest.csv"

pd.set_option("display.max_colwidth", 60)
print("setup ok")


setup ok


## 1) โหลด manifest และเลือกเฉพาะรูปที่มี ground truth (282 รูป: 47 original + 235 augmented)

In [2]:
manifest = pd.read_csv(MANIFEST)
manifest["gt_exists"] = manifest["gt_exists"].astype(str) == "True"
ds = manifest[manifest["gt_exists"]].reset_index(drop=True)

print("total images in manifest:", len(manifest))
print("images with ground truth (used in this notebook):", len(ds))
print(ds.groupby(["split", "group"]).size())


total images in manifest: 288
images with ground truth (used in this notebook): 282
split      group
augmented  G        115
           th       120
original   G         23
           th        24
dtype: int64


## 2) ฟังก์ชัน OCR + Field matching

**OCR:** ใช้ Tesseract (`tha+eng`) — engine ตัวเดียวกับที่มีใน `ocr_system` ของ week3

**Field Level matching:** ground truth เป็นข้อมูลโครงสร้าง (JSON) ส่วน OCR คืนข้อความดิบไม่มีโครงสร้าง
จึงไม่สามารถ "ดึงฟิลด์" แบบ 1:1 ได้ตรง ๆ — ใช้วิธี **fuzzy substring search**: เลื่อนหน้าต่าง (sliding window)
ความยาวเท่ากับค่า ground truth ไปทาบกับข้อความ OCR ทั้งหมด (หลัง normalize ตัดช่องว่างออก เพราะเอกสารไทย
มักไม่มีช่องว่างระหว่างคำ) แล้วหาตำแหน่งที่ Character Error Rate (CER) ต่ำที่สุด ถือว่า field "ถูกต้อง"
เมื่อ CER ≤ 0.2

**ข้อจำกัดที่ทราบล่วงหน้า:** ฟิลด์วันที่ (`date_of_birth`, `admis_date`) ในเอกสารพิมพ์เป็นวันที่ภาษาไทย
(เช่น "3 กันยายน 2542") ขณะที่ ground truth เก็บเป็น ISO (`1999-09-03`) รูปแบบต่างกันโดยธรรมชาติ
คาดว่า field กลุ่มนี้จะได้ CER สูง/ไม่แมตช์ — เป็น**ข้อค้นพบจริง**ไม่ใช่ข้อผิดพลาดของโค้ด


In [3]:
FIELDS = ["student_id", "prename", "name", "faculty_name",
          "degree", "program", "date_of_birth", "admis_date"]
FIELD_MATCH_THRESHOLD = 0.2  # CER <= this => field considered correct


def normalize(s):
    if not s:
        return ""
    return re.sub(r"\s+", "", str(s)).lower()


def best_field_cer(gt_value, ocr_text_norm):
    gt = normalize(gt_value)
    if not gt:
        return None
    if gt in ocr_text_norm:
        return 0.0
    L = len(gt)
    if not ocr_text_norm:
        return 1.0
    step = max(1, L // 8)
    best = 1.0
    for i in range(0, max(1, len(ocr_text_norm) - L + 1), step):
        window = ocr_text_norm[i:i + L]
        d = Levenshtein.distance(gt, window) / max(L, 1)
        if d < best:
            best = d
        if best == 0:
            break
    return min(best, 1.0)


def run_tesseract(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return pytesseract.image_to_string(gray, lang="tha+eng")


def flatten_ground_truth(gt):
    h = gt["header_detail"]
    keys = ["uni_name", "uni_address", "student_id", "faculty_name", "prename",
            "name", "date_of_birth", "admis_date", "grad_date", "grad_reason",
            "degree", "major", "program"]
    return "\n".join(str(h.get(k) or "") for k in keys)


print("functions ready")


functions ready


## 3) รัน OCR + evaluate ทีละรูป (282 รูป) 

In [4]:
t_start = time.time()
field_rows = []
page_rows = []

for i, row in ds.iterrows():
    img_path = f"{DATASET}/{row['image_path']}"
    gt_path = f"{DATASET}/{row['gt_path']}"
    gt = json.load(open(gt_path, encoding="utf-8"))
    header = gt["header_detail"]

    ocr_text = run_tesseract(img_path)
    ocr_norm = normalize(ocr_text)

    # ---- Field Level ----
    for field in FIELDS:
        cer = best_field_cer(header.get(field), ocr_norm)
        if cer is None:
            continue
        field_rows.append({
            "image_path": row["image_path"], "student_id": row["student_id"],
            "group": row["group"], "split": row["split"], "aug_type": row["aug_type"],
            "field": field, "gt_value": header.get(field), "cer": cer,
            "correct": cer <= FIELD_MATCH_THRESHOLD,
        })

    # ---- Page Level ----
    ref_text = flatten_ground_truth(gt)
    page_cer = char_error_rate(ref_text, ocr_text)
    ref_n, hyp_n = normalize_text(ref_text), normalize_text(ocr_text)
    page_wer = wer(ref_n, hyp_n) if ref_n else (0.0 if not hyp_n else 1.0)
    page_rows.append({
        "image_path": row["image_path"], "student_id": row["student_id"],
        "group": row["group"], "split": row["split"], "aug_type": row["aug_type"],
        "page_cer": page_cer, "page_wer": page_wer,
    })

    if (i + 1) % 25 == 0 or (i + 1) == len(ds):
        print(f"  {i+1}/{len(ds)} done  ({time.time()-t_start:.0f}s elapsed)")

print(f"\nTOTAL OCR+eval time: {time.time()-t_start:.1f}s for {len(ds)} images")

field_df = pd.DataFrame(field_rows)
page_df = pd.DataFrame(page_rows)
print("field_df rows:", len(field_df), "| page_df rows:", len(page_df))


  25/282 done  (44s elapsed)


  50/282 done  (99s elapsed)


  75/282 done  (141s elapsed)


  100/282 done  (188s elapsed)


  125/282 done  (220s elapsed)


...

  200/282 done  (321s elapsed)


  225/282 done  (351s elapsed)


  250/282 done  (378s elapsed)


  275/282 done  (411s elapsed)


  282/282 done  (418s elapsed)

TOTAL OCR+eval time: 418.4s for 282 images
field_df rows: 2256 | page_df rows: 282


## 4) Field Level — ความแม่นยำแยกตามฟิลด์

เฉลี่ย CER และ % ที่ถือว่า "ถูกต้อง" (CER ≤ 0.2) ของแต่ละฟิลด์ ข้ามทุกรูป (original+augmented)


In [5]:
field_level = (
    field_df.groupby("field")
    .agg(mean_cer=("cer", "mean"), accuracy_pct=("correct", lambda s: 100 * s.mean()),
         n=("cer", "size"))
    .round(3)
    .sort_values("accuracy_pct", ascending=False)
)
field_level


,mean_cer,accuracy_pct,n
field,,,
student_id,0.002,99.645,282
program,0.012,98.936,282
degree,0.012,98.582,282
prename,0.007,97.872,282
faculty_name,0.039,97.518,282
name,0.037,95.745,282
admis_date,0.594,0.000,282
date_of_birth,0.646,0.000,282


## 5) Page Level — ความแม่นยำระดับทั้งหน้าเอกสาร

เทียบข้อความ OCR ทั้งหมดกับ ground truth ที่ flatten แล้ว (CER/WER) — แยกดูภาพรวม, original vs augmented,
และแยกตามชนิด augmentation (bonus breakdown)


In [6]:
print("=== Overall Page Level ===")
print(page_df[["page_cer", "page_wer"]].mean().round(3))

print("\n=== แยก original vs augmented ===")
display(page_df.groupby("split")[["page_cer", "page_wer"]].mean().round(3))

print("\n=== แยกตามชนิด augmentation ===")
display(page_df.groupby("aug_type")[["page_cer", "page_wer"]].mean().round(3).sort_values("page_cer"))


=== Overall Page Level ===
page_cer     5.576
page_wer    69.988
dtype: float64

=== แยก original vs augmented ===


,page_cer,page_wer
split,,
augmented,5.532,69.477
original,5.793,72.546



=== แยกตามชนิด augmentation ===


,page_cer,page_wer
aug_type,,
blur_contrast,4.489,59.890
rotate_bright,5.684,72.677
noise,5.791,71.426
none,5.793,72.546
perspective,5.801,71.571
jpeg_dark,5.896,71.819


## 6) Category Level — แยกตามระดับปริญญา (`th` = ป.ตรี, `G` = บัณฑิตศึกษา)

รวมผลทั้ง Field Level และ Page Level แล้วเฉลี่ยแยกตาม category


In [7]:
category_field = (
    field_df.groupby("group")
    .agg(mean_field_cer=("cer", "mean"), field_accuracy_pct=("correct", lambda s: 100 * s.mean()))
    .round(3)
)
category_page = page_df.groupby("group")[["page_cer", "page_wer"]].mean().round(3)

category_level = category_field.join(category_page)
category_level.index = category_level.index.map({"th": "th (ป.ตรี)", "G": "G (บัณฑิตศึกษา)"})
category_level


,mean_field_cer,field_accuracy_pct,page_cer,page_wer
group,,,,
G (บัณฑิตศึกษา),0.171,73.370,4.272,50.147
th (ป.ตรี),0.166,73.698,6.825,89.002


## 7) บันทึกผลลัพธ์เป็นไฟล์ 

In [8]:
os.makedirs("results", exist_ok=True)
field_df.to_csv("results/field_level_raw.csv", index=False)
page_df.to_csv("results/page_level_raw.csv", index=False)
field_level.to_csv("results/field_level_summary.csv")
category_level.to_csv("results/category_level_summary.csv")
page_df.groupby("split")[["page_cer", "page_wer"]].mean().round(3).to_csv("results/page_level_by_split.csv")
page_df.groupby("aug_type")[["page_cer", "page_wer"]].mean().round(3).to_csv("results/page_level_by_augtype.csv")
print("saved results/*.csv")


saved results/*.csv


## สรุป

- รัน OCR (Tesseract, `tha+eng`) บน **282 รูป** จาก `Lab5_transcript_dataset` (47 original + 235 augmented ที่มี ground truth)
- **Field Level**: ฟิลด์ข้อความ (ชื่อ, คณะ, ปริญญา) แม่นกว่าฟิลด์วันที่มาก เพราะรูปแบบวันที่ในเอกสาร (ภาษาไทย)
  กับ ground truth (ISO) ไม่ตรงกันโดยธรรมชาติ ไม่ใช่ความผิดพลาดของ OCR
- **Page Level**: ภาพ augmented (โดยเฉพาะ `blur_contrast`, `jpeg_dark`, `perspective`) มี CER/WER สูงกว่า original
  อย่างชัดเจน สอดคล้องกับที่ augmentation จำลองสภาพเอกสารที่อ่านยากขึ้น
- **Category Level**: เทียบความแม่นยำระหว่างเอกสารระดับ ป.ตรี (`th`) กับบัณฑิตศึกษา (`G`)

**ข้อจำกัดของวิธีวัด Field Level:** ใช้ fuzzy substring matching แทนการดึงฟิลด์แบบมีโครงสร้างจริง
(ไม่มี layout/bounding-box parsing ในเวลาที่จำกัด) ตัวเลขจึงเป็นค่าประมาณความแม่นยำ ไม่ใช่ field extraction ที่สมบูรณ์
